<div style="border-radius: 10px; padding: 32px 0px; border: 1px solid rgba(128,128,128,0.2);">
  <div style="display: flex; align-items: center; gap: 16px; padding: 0 32px;">
    <div>
      <h1 style="margin:0; font-size:1.6em;">B1 · Clasificador Hídrico y Natural</h1>
      <p style="margin:4px 0 0; color: gray;">Clasificador multietiqueta de publicaciones hídricas y naturales en boletines oficiales españoles</p>
    </div>
  </div>
</div>


# B1 · Clasificador Hídrico y Natural

Este notebook implementa el clasificador del **bloque B1** (hídrico y natural) para el proyecto de monitorización de boletines oficiales españoles.

**Dominio**: publicaciones relacionadas con concesiones de aguas, vías pecuarias, montes de utilidad pública, espacios naturales protegidos, gestión de residuos, vertidos y planes hidrológicos.

**Arquitectura**: B1 comparte con B0 el pre-procesamiento N0/N1 (`agent.py`) pero tiene su propio schema (`schema_B1.py`) y prompts (`prompts_B1.py`).

| Capa | Qué etiqueta | Cómo |
|------|-------------|------|
| N0 | Ámbito geográfico | lookup por boletín, sin LLM |
| N1 | Tipo de acto | reglas de primer token, sin LLM |
| N2 | Categoría hídrica/natural | LLM (11 etiquetas) |
| N3 | Subcategoría uso/cuenca | LLM (15 etiquetas) |


---

##  0. Setup

Cargamos variables de entorno e importamos librerías. El modelo vive en LM Studio.

In [1]:
# Librerías estándar
import os
import html
import re
import json
import asyncio
from pathlib import Path

# Datos
import pandas as pd
import numpy as np

# Métricas multilabel
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    jaccard_score,
)

# Carga de variables de entorno desde .env (busca hacia arriba desde el directorio actual)
from dotenv import find_dotenv, load_dotenv

# Pydantic AI - framework que fuerza al LLM a devolver JSON validado por Pydantic
from pydantic_ai import Agent

SEED = 29092025

load_dotenv(find_dotenv())


True

In [2]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

# Cambiar LM_STUDIO_MODEL según el modelo cargado en LM Studio
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)


/tmp/ipykernel_1857476/2598252738.py:7: DeprecationWarning: `OpenAIModel` was renamed to `OpenAIChatModel` to clearly distinguish it from `OpenAIResponsesModel` which uses OpenAI's newer Responses API. Use that unless you're using an OpenAI Chat Completions-compatible API, or require a feature that the Responses API doesn't support yet like audio.
  model = OpenAIModel(


In [3]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado en LM Studio"
    print(f"OK: {LM_STUDIO_MODEL} listo  |  otros: {[m for m in modelos if m != LM_STUDIO_MODEL]}")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde - arrancar el servidor antes de continuar")


OK: qwen/qwen3.5-9b listo  |  otros: ['google/gemma-4-e4b', 'google/gemma-3n-e4b', 'gemma-4-e2b-it', 'google/gemma-3-4b', 'text-embedding-nomic-embed-text-v1.5']


---

##  1. Schema de output

El schema define el **contrato entre el LLM y el sistema**: qué campos devuelve el modelo,
de qué tipo son y qué invariantes deben cumplirse.

B1 reutiliza `ActType` de B0 (mismo enum de formas jurídicas) y define sus propios enums
`CategoryType` (N2) y `SubcategoryType` (N3).


In [4]:
# Importar los tipos del schema B1: enums de etiquetas y modelo de output
from clasificador.schema_B1 import ActType, CategoryType, SubcategoryType, ClassifierOutput


In [5]:
# Verificación de invariantes del schema B1
ejemplo_valido = ClassifierOutput(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    categories=[CategoryType.AGU_RIE],
    subcategories=[SubcategoryType.USO_AGRICOLA, SubcategoryType.CUENCA_DUERO],
    confidence=0.95,
    reasoning="'concesión de aguas para riego' → AGU_RIE. 'Confederación Hidrográfica del Duero' → cuenca_duero.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutput(
        is_relevant=False, act_type=ActType.RESOLUCION,
        categories=[CategoryType.AGU_GEN], subcategories=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError -> {e.errors()[0]['msg']}")


Ejemplo válido:
{
  "is_relevant": true,
  "act_type": "resolución",
  "categories": [
    "AGU_RIE"
  ],
  "subcategories": [
    "uso_agricola",
    "cuenca_duero"
  ],
  "confidence": 0.95,
  "reasoning": "'concesión de aguas para riego' → AGU_RIE. 'Confederación Hidrográfica del Duero' → cuenca_duero."
}

Violación de invariante:
  ValidationError -> Value error, is_relevant=False con categories != []


---

##  2. Pre-procesamiento

El pre-procesamiento N0/N1 es **compartido entre todos los bloques** y vive en `agent.py`.
No se reimplementa aquí: solo se importa.

- **N0**: ámbito geográfico por boletín (`get_ambito`)
- **N1**: tipo de acto por reglas de primer token (`inferir_act_type`)


In [6]:
# Funciones N0/N1 compartidas con B0 - viven en agent.py
from clasificador.agent import get_ambito, inferir_act_type

# Cargar el corpus completo
PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"
df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)
print(f"Corpus total: {len(df):,} registros | Columnas: {list(df.columns)}")


Corpus total: 65,201 registros | Columnas: ['id', 'pdf_link', 'expediente', 'promotor', 'proyecto', 'description', 'tipo', 'clean_id', 'bulletin', 'provincias', 'municipios', 'raw_scraped_timestamp', 'raw_scraped_year_month', 'scraped_timestamp', 'scraped_year_month', 'publication_timestamp', 'publication_type', 'contains_aau', 'proxy_pdf_link', 'ambito', 'rango']


In [7]:
# Aplicar clasificador N1 (reglas de primer token) al corpus completo
df["act_type_n1"] = df.apply(
    lambda r: inferir_act_type(str(r["description"]), str(r["bulletin"])).value, axis=1
)

# Filtrar al dominio B1 usando las keywords de muestreo para estimar el tamaño del universo
keywords_dominio = [
    "confederación hidrográfica", "comisaría de aguas", "aprovechamiento de aguas",
    "regadío", "comunidad de regantes", "riego", "sondeo para captación",
    "captación de aguas subterráneas", "abastecimiento de agua", "abastecimiento municipal",
    "aprovechamiento hidroeléctrico", "central hidroeléctrica",
    "vía pecuaria", "cañada real", "cordel", "vereda",
    "monte de utilidad pública", "dominio público forestal",
    "parque natural", "parque nacional", "red natura", "zepa", "zec",
    "gestión de residuos", "tratamiento de residuos",
    "autorización de vertido", "vertido de aguas",
    "plan hidrológico", "demarcación hidrográfica",
]
mask_b1 = df["description"].str.lower().str.contains("|".join(keywords_dominio), na=False)
df_b1 = df[mask_b1].copy()
print(f"Universo B1 estimado: {len(df_b1):,} registros ({len(df_b1)/len(df)*100:.2f}% del corpus)")
print(f"\nDistribución N1 en universo B1:")
print(df_b1["act_type_n1"].value_counts().head(10).to_string())


Universo B1 estimado: 3,168 registros (4.86% del corpus)

Distribución N1 en universo B1:
act_type_n1
anuncio                2388
resolución              544
información_pública      87
aprobación               29
edicto                   26
orden                    20
acuerdo                  15
otros                    15
notificación             11
corrección_errores       10


---

##  3. Ground truth — Muestreo estratificado

El ground truth se construye en dos pasos:
1. **Muestreo**: seleccionar registros representativos por categoría N2 (esta sección)
2. **Anotación**: etiquetar manualmente el CSV resultante (tarea externa)

### Cuotas de muestreo

| Categoría | Cuota | Señal principal |
|-----------|-------|----------------|
| AGU_GEN | 25 | confederación hidrográfica, aprovechamiento de aguas |
| AGU_RIE | 25 | regadío, comunidad de regantes |
| AGU_SND | 15 | sondeo para captación |
| AGU_ABS | 15 | abastecimiento de agua |
| VIA_PEC | 20 | vía pecuaria, cañada real |
| RES | 15 | gestión de residuos |
| MON | 10 | monte de utilidad pública |
| VER | 10 | autorización de vertido |
| ESP_NAT | 5 | parque natural, Red Natura, ZEPA |
| AGU_IND | 5 | aprovechamiento hidroeléctrico |
| PHD | 5 | plan hidrológico (si hay pool suficiente) |
| **Negativos** | **20** | sin ninguna de las keywords anteriores |


In [8]:
# Pre-flight check: verificar pool disponible por label antes de samplear.
# Si alguna etiqueta tiene menos registros que la cuota, se reduce automaticamente.
import random
random.seed(SEED)

keywords_B1 = {
    "AGU_GEN": ["confederación hidrográfica", "comisaría de aguas", "aprovechamiento de aguas"],
    "AGU_RIE": ["regadío", "comunidad de regantes", "riego"],
    "AGU_SND": ["sondeo para captación", "captación de aguas subterráneas", "pozo de captación"],
    "AGU_ABS": ["abastecimiento de agua", "abastecimiento municipal", "agua potable"],
    "AGU_IND": ["aprovechamiento hidroeléctrico", "central hidroeléctrica", "refrigeración"],
    "VIA_PEC": ["vía pecuaria", "cañada real", "cordel", "vereda", "colada"],
    "MON":     ["monte de utilidad pública", "dominio público forestal", "ocupación de monte"],
    # ESP_NAT: keywords corregidas - "espacio natural protegido" y "reserva natural"
    # no devuelven resultados en el corpus Q1 2025
    "ESP_NAT": ["parque natural", "parque nacional", "red natura", "zepa", "zec", "zona de especial"],
    "RES":     ["gestión de residuos", "tratamiento de residuos", "planta de residuos"],
    "VER":     ["autorización de vertido", "vertido de aguas residuales", "dominio público hidráulico"],
    "PHD":     ["plan hidrológico", "demarcación hidrográfica"],
}

cuotas = {
    "AGU_GEN": 25, "AGU_RIE": 25, "AGU_SND": 15, "AGU_ABS": 15,
    "AGU_IND": 5,  "VIA_PEC": 20, "MON": 10,      "ESP_NAT": 5,
    "RES": 15,     "VER": 10,     "PHD": 5,
}
N_NEGATIVOS = 20

print(f"{'Label':<10} {'Pool':>8} {'Cuota':>7} {'Disponible':>11}")
print("-" * 42)
for label, kws in keywords_B1.items():
    mask = df["description"].str.lower().str.contains("|".join(kws), na=False)
    pool_size = mask.sum()
    cuota = cuotas.get(label, 0)
    disponible = min(cuota, pool_size)
    estado = "OK" if pool_size >= cuota else f"REDUCIDA a {disponible}"
    print(f"{label:<10} {pool_size:>8,} {cuota:>7} {estado:>11}")


Label          Pool   Cuota  Disponible
------------------------------------------
AGU_GEN       1,428      25          OK
AGU_RIE       1,147      25          OK
AGU_SND         170      15          OK
AGU_ABS         267      15          OK
AGU_IND          23       5          OK
VIA_PEC         307      20          OK
MON              90      10          OK
ESP_NAT          39       5          OK
RES             142      15          OK
VER             131      10          OK
PHD              30       5          OK


In [9]:
# Muestreo estratificado por categoria.
# min(cuota, len(pool)) evita errores cuando el pool es menor que la cuota.
sampled_ids = set()
frames = []

for label, kws in keywords_B1.items():
    mask = df["description"].str.lower().str.contains("|".join(kws), na=False)
    mask = mask & ~df.index.isin(sampled_ids)
    pool = df[mask]
    n = min(cuotas.get(label, 0), len(pool))
    if n == 0:
        print(f"  {label:<10} SKIP (pool vacio)")
        continue
    sample = pool.sample(n, random_state=SEED).copy()
    sample["grupo_muestreo"] = label
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {label:<10} pool={len(pool):>5,}  sampled={n}")

# Negativos: registros sin ninguna keyword del dominio B1
all_kws = [kw for kws in keywords_B1.values() for kw in kws]
mask_neg = ~df["description"].str.lower().str.contains("|".join(all_kws), na=False)
mask_neg = mask_neg & ~df.index.isin(sampled_ids)
negativos = df[mask_neg].sample(N_NEGATIVOS, random_state=SEED).copy()
negativos["grupo_muestreo"] = "NEGATIVO"
frames.append(negativos)

df_muestreo = pd.concat(frames, ignore_index=True)
df_muestreo["id"] = range(len(df_muestreo))
print(f"\nTotal muestreado: {len(df_muestreo)} registros")


  AGU_GEN    pool=1,428  sampled=25
  AGU_RIE    pool=1,143  sampled=25
  AGU_SND    pool=  169  sampled=15
  AGU_ABS    pool=  266  sampled=15
  AGU_IND    pool=   23  sampled=5
  VIA_PEC    pool=  306  sampled=20
  MON        pool=   90  sampled=10
  ESP_NAT    pool=   39  sampled=5
  RES        pool=  142  sampled=15
  VER        pool=  130  sampled=10
  PHD        pool=   30  sampled=5

Total muestreado: 170 registros


In [10]:
# Validación del muestreo: 3 ejemplos por label para confirmar que pertenecen al dominio.
# Imprime el grupo, boletín y descripción truncada para revisión rápida.
print("Validación del muestreo - 3 ejemplos por grupo:\n")
for grupo in df_muestreo["grupo_muestreo"].unique():
    muestra = df_muestreo[df_muestreo["grupo_muestreo"] == grupo].head(3)
    print(f"--- {grupo} ---")
    for _, row in muestra.iterrows():
        desc = str(row["description"])[:130].replace("\n", " ")
        bul = str(row.get("bulletin", "?")).upper()
        print(f"  [{bul}] {desc}")
    print()


Validación del muestreo - 3 ejemplos por grupo:

--- AGU_GEN ---
  [BOE] Anuncio de la Confederación Hidrográfica del Guadiana de sometimiento a información pública de resolución de otorgamiento de conce
  [BOE] Anuncio de formalización de contratos de: Presidencia de la Confederación Hidrográfica del Guadiana. Objeto: Suministro de materia
  [BOE] Anuncio de formalización de contratos de: Presidencia de la Confederación Hidrográfica del Guadalquivir. Objeto: Contrato de servi

--- AGU_RIE ---
  [BOE] Anuncio de la Comunidad de Regantes Pozo La Sierra de Torrent sobre convocatoria de Junta.
  [BOE] Anuncio de la Comisaría de Aguas de la Confederación Hidrográfica del Guadalquivir de información pública de procedimiento de exti
  [BOE] Anuncio de la Comunidad de Regantes Las Almacidas sobre convocatoria de Junta General Ordinaria.

--- AGU_SND ---
  [DOGV] RESOLUCIÓN de 17 de septiembre de 2024, de la Dirección General de Urbanismo, Paisaje y Evaluación Ambiental, por la que se formul
 

In [11]:
# Guardar el muestreo sin anotar - la anotación manual es el siguiente paso
PATH_MUESTREO = "../data/ground_truth/ground_truth_B1_muestreo.csv"
Path(PATH_MUESTREO).parent.mkdir(parents=True, exist_ok=True)

# Columnas de anotación preparadas vacías para rellenar manualmente
df_muestreo["is_relevant_gt"]   = ""
df_muestreo["categories_gt"]    = ""
df_muestreo["subcategories_gt"] = ""
df_muestreo["notas_anotador"]   = ""

df_muestreo[["id","bulletin","description","grupo_muestreo",
             "is_relevant_gt","categories_gt","subcategories_gt","notas_anotador"]].to_csv(
    PATH_MUESTREO, index=False
)
print(f"Guardado: {PATH_MUESTREO}  ({len(df_muestreo)} registros)")
print("Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt")


Guardado: ../data/ground_truth/ground_truth_B1_muestreo.csv  (170 registros)
Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt


### Dataset anotado

Una vez completada la anotación manual de `ground_truth_B1_muestreo.csv`, cargar con:

```python
df_anotado = pd.read_csv("../data/ground_truth/ground_truth_B1_anotado.csv")
```

La anotación debe cubrir las columnas `is_relevant_gt`, `categories_gt` (valores separados por coma, ej. `AGU_RIE,ESP_NAT`) y `subcategories_gt` (opcional).


---

##  4. Agente base

La lógica de clasificación vive en `src/clasificador/`. El notebook solo importa y orquesta.
`build_agent` acepta `output_type` y `prompt_registry` para soportar bloques distintos de B0.


In [12]:
# Importar la capa de agente compartida y los prompts/schema propios de B1
from clasificador.agent import build_agent, run_experiment, clasificar_async
from clasificador.prompts_B1 import SYSTEM_PROMPT_B1_V1, PROMPT_REGISTRY as PROMPT_REGISTRY_B1
from clasificador.schema_B1 import ClassifierOutput as ClassifierOutputB1
from tqdm.asyncio import tqdm_asyncio


In [13]:
# Construir el agente B1 con el prompt baseline (v1)
# Se pasan output_type y prompt_registry propios de B1 - agent.py permanece genérico
agent_b1_v1 = build_agent(
    model, "v1",
    output_type=ClassifierOutputB1,
    prompt_registry=PROMPT_REGISTRY_B1,
)


In [ ]:
# Validación cualitativa - 5 casos representativos del dominio B1
casos_b1 = [
    ("Resolución de la Confederación Hidrográfica del Duero por la que se otorga "
     "concesión de aguas superficiales para riego de 120 ha en Valladolid. "
     "Comunidad de Regantes del Canal de Macías Picavea.", "bocyl"),
    ("Anuncio de información pública sobre solicitud de autorización de vía pecuaria "
     "cañada real para construcción de línea eléctrica subterránea en Cáceres.", "doe"),
    ("Resolución de la Dirección General de Medio Natural por la que se aprueba "
     "el plan de gestión de la Zona de Especial Conservación (ZEC) ES4110108.", "bocyl"),
    ("Resolución de la Consejería de Hacienda por la que se convocan plazas de "
     "auxiliar administrativo en la Junta de Castilla y León.", "bocyl"),
    ("Anuncio de la Confederación Hidrográfica del Tajo sobre autorización de "
     "vertido de aguas residuales tratadas al río Jarama en Guadalajara.", "bocm"),
]

print("Validación cualitativa - 5 casos B1\n")
for i, (desc, bul) in enumerate(casos_b1, 1):
    msg = f"Boletín: {bul.upper()} (ámbito: {get_ambito(bul)})\n\nDescripción: {desc}"
    result = await agent_b1_v1.run(msg)
    r = result.output
    cats = [c.value for c in r.categories]
    subs = [s.value for s in r.subcategories]
    print(f"Caso {i} | relevant={r.is_relevant} | cats={cats} | subs={subs}")
    print(f"  Reasoning: {r.reasoning[:120]}")
    print()


Validación cualitativa - 5 casos B1



---

##  5. Experimento 1 - Baseline zero-shot

**Problema**: no existe un clasificador para el dominio hídrico/natural. Necesitamos una línea base que mida el rendimiento zero-shot antes de cualquier optimización.

**Objetivo**: establecer el Macro-F1 de referencia con el prompt mínimo operativo (V1) y detectar los patrones de error sistemáticos que guiarán la siguiente iteración del prompt.

**Enfoque**: Qwen 3.5 9B · `SYSTEM_PROMPT_B1_V1` · zero-shot · sin contexto N1 · 180 registros.

**Resultados**:

| Métrica | Valor |
|---------|-------|
| is_rel F1 | - |
| Micro F1 | - |
| Macro F1 | - |
| Hamming Loss | - |
| Jaccard | - |
| Subset Acc | - |


In [14]:

df_anotado = pd.read_csv("../data/ground_truth/ground_truth_B1_anotado.csv")
df_anotado_run = df_anotado[df_anotado["is_relevant_gt"] != ""]  # solo filas anotadas

df_exp_b1_1 = await run_experiment(
     agent_b1_v1, df_anotado_run,
     use_n1_context=False, concurrency=1,
     output_path="../results/b1_exp1_baseline_qwen9b.csv",
     desc="B1 Exp1 - Baseline",
 )


  Reanudando: 170/170 registros ya clasificados


---

##  6. Análisis de errores - Baseline

Evaluamos las predicciones del Exp 1 contra el ground truth para identificar patrones de error
que guíen la mejora del prompt en §7.


In [15]:
# Convierte cualquier representación de etiquetas a un set de strings.
# Misma función que B0 - compatible con CSV manual ("AGU_RIE,ESP_NAT")
# y con JSON de predicciones ('["AGU_RIE","ESP_NAT"]').
def parse_labels(value) -> set:
    if pd.isna(value) or str(value).strip() in ("", "nan"):
        return set()
    v = str(value).strip()
    if v.startswith("["):
        try:
            return set(json.loads(v))
        except Exception:
            pass
    return set(x.strip() for x in v.split(",") if x.strip())


In [16]:
def compute_metrics_B1(df_eval, label="", verbose=True):
    """
    Calcula métricas multilabel para el clasificador B1.
    Identica a compute_metrics de B0 pero con las etiquetas N2 de B1.
    Columnas esperadas: is_relevant_gt, categories_gt, is_relevant_pred, categories_pred.
    """
    # Etiquetas N2 de B1 - derivadas del enum para evitar hardcodear
    from clasificador.schema_B1 import CategoryType
    N2 = [e.value for e in CategoryType]

    y_true = df_eval["is_relevant_gt"].astype(bool)
    y_pred = df_eval["is_relevant_pred"].fillna(False).astype(bool)

    # -- Subproblema binario: is_relevant --
    tp = ((y_true) & (y_pred)).sum()
    fp = ((~y_true) & (y_pred)).sum()
    fn = ((y_true) & (~y_pred)).sum()
    tn = ((~y_true) & (~y_pred)).sum()
    prec_rel = tp / (tp + fp) if tp + fp > 0 else 0.0
    rec_rel  = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1_rel   = 2 * prec_rel * rec_rel / (prec_rel + rec_rel) if prec_rel + rec_rel > 0 else 0.0
    acc_rel  = accuracy_score(y_true, y_pred)

    if verbose:
        if label:
            print(f"-- {label} --")
        print(f"\nis_relevant  Acc={acc_rel:.3f}  P={prec_rel:.3f}  R={rec_rel:.3f}  F1={f1_rel:.3f}")
        print(f"             TP={tp}  FP={fp}  FN={fn}  TN={tn}\n")

    # -- Multilabel N2: construir matrices binarias --
    Y_true = np.array([
        [1 if lbl in parse_labels(r["categories_gt"])   else 0 for lbl in N2]
        for _, r in df_eval.iterrows()
    ])
    Y_pred = np.array([
        [1 if lbl in parse_labels(r["categories_pred"]) else 0 for lbl in N2]
        for _, r in df_eval.iterrows()
    ])

    micro_f1   = f1_score(Y_true, Y_pred, average="micro",    zero_division=0)
    macro_f1   = f1_score(Y_true, Y_pred, average="macro",    zero_division=0)
    hl         = hamming_loss(Y_true, Y_pred)
    jaccard    = jaccard_score(Y_true, Y_pred, average="samples", zero_division=0)
    subset_acc = accuracy_score(Y_true, Y_pred)

    if verbose:
        print(f"N2 multilabel:")
        print(f"  Micro F1:      {micro_f1:.3f}")
        print(f"  Macro F1:      {macro_f1:.3f}")
        print(f"  Hamming Loss:  {hl:.4f}")
        print(f"  Jaccard:       {jaccard:.3f}")
        print(f"  Subset Acc:    {subset_acc:.3f}\n")

        print(f"{'Label':<10} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5}")
        print("-" * 35)
        for i, lbl in enumerate(N2):
            p2  = precision_score(Y_true[:, i], Y_pred[:, i], zero_division=0)
            r2  = recall_score   (Y_true[:, i], Y_pred[:, i], zero_division=0)
            f12 = f1_score       (Y_true[:, i], Y_pred[:, i], zero_division=0)
            print(f"{lbl:<10} {p2:>6.3f} {r2:>6.3f} {f12:>6.3f} {Y_true[:, i].sum():>5}")
        print("-" * 35)
        print(f"{'Macro':<10} {'':>13} {macro_f1:>6.3f}")

    card      = Y_true.sum(axis=1)
    exact_arr = (Y_true == Y_pred).all(axis=1)
    exact_ser = pd.Series(exact_arr, index=df_eval.index)

    if verbose:
        print("\nSubset Acc por cardinalidad:")
        for c, lbl_c in [(0, "card=0 (no relevante)"), (1, "card=1"), (2, "card=2"), (3, "card>=3")]:
            mask = (card >= c) if c == 3 else (card == c)
            if mask.sum() > 0:
                print(f"  {lbl_c:<22}: {exact_arr[mask].mean():.3f}  ({exact_arr[mask].sum()}/{mask.sum()})")
        conf = df_eval["confidence"].dropna()
        print(f"\nConfianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")

    return {
        "is_rel_accuracy":  float(acc_rel),
        "is_rel_precision": float(prec_rel),
        "is_rel_recall":    float(rec_rel),
        "is_rel_f1":        float(f1_rel),
        "micro_f1":         float(micro_f1),
        "macro_f1":         float(macro_f1),
        "hamming_loss":     float(hl),
        "jaccard_samples":  float(jaccard),
        "subset_accuracy":  float(subset_acc),
        "exact":            exact_ser,
        "rel":              y_true,
    }


In [17]:
# Muestra los registros relevantes donde la predicción N2 no coincide con el GT.
def print_errors(df_eval, exact, rel, label="", n=None):
    errores = df_eval[~exact & rel]
    if n is not None:
        errores = errores.head(n)
    print(f"-- Errores N2 en relevantes{' · ' + label if label else ''} --")
    print(f"Total: {(~exact & rel).sum()}\n")
    for _, row in errores.iterrows():
        gt   = parse_labels(row["categories_gt"])
        pred = parse_labels(row["categories_pred"])
        print(f"ID {row['id']} | GT={sorted(gt)} | PRED={sorted(pred)}")
        print(f"  Falta: {sorted(gt - pred)} | Sobra: {sorted(pred - gt)}")
        print(f"  {str(row['description'])[:100]}...")
        print(f"  Razonamiento: {str(row.get('reasoning', ''))[:150]}")
        print()


In [18]:
df_b1_exp1 = pd.read_csv("../results/b1_exp1_baseline_qwen9b.csv")
df_eval_b1_1 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b1_exp1[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b1_1 = compute_metrics_B1(df_eval_b1_1, "Experimento B1-1 - Baseline")


-- Experimento B1-1 - Baseline --

is_relevant  Acc=0.818  P=0.818  R=1.000  F1=0.900
             TP=139  FP=31  FN=0  TN=0

N2 multilabel:
  Micro F1:      0.859
  Macro F1:      0.864
  Hamming Loss:  0.0235
  Jaccard:       0.675
  Subset Acc:    0.788

Label           P      R     F1   Sup
-----------------------------------
AGU_GEN     0.579  0.611  0.595    18
AGU_RIE     1.000  0.871  0.931    31
AGU_SND     0.739  0.895  0.810    19
AGU_ABS     0.909  0.952  0.930    21
AGU_IND     0.750  0.857  0.800     7
VIA_PEC     0.952  0.952  0.952    21
MON         1.000  0.900  0.947    10
ESP_NAT     0.417  1.000  0.588     5
RES         1.000  0.909  0.952    11
VER         1.000  1.000  1.000     7
PHD         1.000  1.000  1.000     2
-----------------------------------
Macro                     0.864

Subset Acc por cardinalidad:
  card=0 (no relevante) : 0.903  (28/31)
  card=1                : 0.766  (98/128)
  card=2                : 0.889  (8/9)
  card>=3               : 0.00

In [19]:
print_errors(df_eval_b1_1, m_b1_1["exact"], m_b1_1["rel"], label="B1 Experimento 1 - Baseline")


-- Errores N2 en relevantes · B1 Experimento 1 - Baseline --
Total: 33

ID 0 | GT=['AGU_GEN'] | PRED=['AGU_SND']
  Falta: ['AGU_GEN'] | Sobra: ['AGU_SND']
  Anuncio de la Confederación Hidrográfica del Guadiana de sometimiento a información pública de resol...
  Razonamiento: El texto menciona explícitamente "concesión de aguas subterráneas" (señal AGU_SND), y por el término municipal de Bolaños de Calatrava en Ciudad Real 

ID 4 | GT=['AGU_RIE'] | PRED=['AGU_GEN', 'AGU_RIE']
  Falta: [] | Sobra: ['AGU_GEN']
  Anuncio de la Confederación Hidrográfica del Duero, O.A., de información pública del expediente de m...
  Razonamiento: La descripción menciona "concesión de un aprovechamiento de aguas subterráneas" (AGU_GEN) y específicamente indica "con destino a riego" (AGU_RIE), ad

ID 5 | GT=['AGU_RIE'] | PRED=['AGU_RIE', 'AGU_SND']
  Falta: [] | Sobra: ['AGU_SND']
  Anuncio de la Confederación Hidrográfica del Duero, O.A., de resolución del expediente de concesión ...
  Razonamiento: El te

---

##  7. Experimento 2 - Prompt v2

**Problema**: el Exp 1 comete errores sistematicos en 6 patrones: (1) AGU_GEN se añade junto a etiquetas especificas de uso, (2) AGU_SND se activa por "aguas subterraneas" aunque el uso sea riego, (3) ESP_NAT se dispara por señales falsas (DG Sostenibilidad, Canarias, vias pecuarias), (4) convocatorias de comunidades de regantes clasificadas como irrelevantes, (5) autorizaciones de uso del DPH (apicultura, tala, obras) etiquetadas como VER, (6) VIA_PEC no detectada cuando solo aparece en el nombre del proyecto.

**Objetivo**: verificar si las reglas explicitas del prompt V2 corrigen estos patrones sin degradar el rendimiento general.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B1_V2 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [20]:
from clasificador.prompts_B1 import PROMPT_REGISTRY as PROMPT_REGISTRY_B1

agent_b1_v2 = build_agent(model, "v2", output_type=ClassifierOutputB1, prompt_registry=PROMPT_REGISTRY_B1)

df_b1_exp2 = await run_experiment(
    agent_b1_v2, df_anotado,
    use_n1_context=False, concurrency=1,
    output_path="../results/b1_exp2_promptv2_qwen9b.csv",
    desc="B1 Exp2 - Prompt v2",
)
df_b1_exp2.head(3)

  Reanudando: 170/170 registros ya clasificados


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s
0,0,boe,Anuncio de la Confederación Hidrográfica del G...,AGU_GEN,True,AGU_SND,cuenca_guadiana,NaN,True,anuncio,"[""AGU_SND""]","[""cuenca_guadiana"", ""uso_agricola""]",0.85,"El texto menciona explícitamente ""concesión de...",168.252
1,1,boe,Anuncio de formalización de contratos de: Pres...,AGU_GEN,False,NaN,NaN,NaN,False,anuncio,[],[],0.98,El texto describe un contrato de suministro y ...,61.812
2,2,boe,Anuncio de formalización de contratos de: Pres...,AGU_GEN,False,NaN,NaN,NaN,False,anuncio,[],[],1.00,La descripción corresponde a un anuncio de for...,69.076


In [21]:
df_b1_exp2 = pd.read_csv("../results/b1_exp2_promptv2_qwen9b.csv")
df_eval_b1_2 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b1_exp2[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b1_2 = compute_metrics_B1(df_eval_b1_2, "Experimento B1-2 - Prompt v2")

-- Experimento B1-2 - Prompt v2 --

is_relevant  Acc=0.818  P=0.818  R=1.000  F1=0.900
             TP=139  FP=31  FN=0  TN=0

N2 multilabel:
  Micro F1:      0.821
  Macro F1:      0.830
  Hamming Loss:  0.0299
  Jaccard:       0.658
  Subset Acc:    0.771

Label           P      R     F1   Sup
-----------------------------------
AGU_GEN     0.435  0.556  0.488    18
AGU_RIE     1.000  0.935  0.967    31
AGU_SND     0.682  0.789  0.732    19
AGU_ABS     0.818  0.857  0.837    21
AGU_IND     0.667  0.857  0.750     7
VIA_PEC     1.000  0.952  0.976    21
MON         0.900  0.900  0.900    10
ESP_NAT     0.556  1.000  0.714     5
RES         1.000  0.727  0.842    11
VER         1.000  0.857  0.923     7
PHD         1.000  1.000  1.000     2
-----------------------------------
Macro                     0.830

Subset Acc por cardinalidad:
  card=0 (no relevante) : 0.871  (27/31)
  card=1                : 0.766  (98/128)
  card=2                : 0.556  (5/9)
  card>=3               : 0.5

In [22]:
print_errors(df_eval_b1_2, m_b1_2["exact"], m_b1_2["rel"], label="B1 Experimento 2 - Prompt v2")

-- Errores N2 en relevantes · B1 Experimento 2 - Prompt v2 --
Total: 35

ID 0 | GT=['AGU_GEN'] | PRED=['AGU_SND']
  Falta: ['AGU_GEN'] | Sobra: ['AGU_SND']
  Anuncio de la Confederación Hidrográfica del Guadiana de sometimiento a información pública de resol...
  Razonamiento: El texto menciona explícitamente "concesión de aguas subterráneas" para un uso en el término municipal, activando AGU_SND (sondeo/captación) y cuenca_

ID 3 | GT=['AGU_GEN'] | PRED=['AGU_SND']
  Falta: ['AGU_GEN'] | Sobra: ['AGU_SND']
  Anuncio de la Confederación Hidrográfica del Júcar, O.A. sobre información pública de una concesión ...
  Razonamiento: El texto es un "Anuncio de la Confederación Hidrográfica del Júcar" sobre "concesión de aguas subterráneas", lo que activa la categoría AGU_SND por se

ID 4 | GT=['AGU_RIE'] | PRED=['AGU_SND']
  Falta: ['AGU_RIE'] | Sobra: ['AGU_SND']
  Anuncio de la Confederación Hidrográfica del Duero, O.A., de información pública del expediente de m...
  Razonamiento: "El text

---

##  8. Experimento 3 - Prompt v3

**Problema**: el Exp 2 sigue fallando en 8 patrones: AGU_SND spurious cuando no hay captacion/sondeo explicito, RES no detectado en IIA de plantas de residuos, VER con "solicitud de autorizacion de vertido" sin "aguas residuales", ESP_NAT falso en centrales y Canarias, AGU_GEN spurious coexistiendo con VIA_PEC/MON/VER, AGU_ABS con señales incompletas, y autorizaciones de uso de cauce sin vertido no reconocidas como AGU_GEN.

**Objetivo**: verificar si las 8 reglas refinadas del prompt V3 corrigen los patrones residuales sin degradar lo que ya funcionaba en V2.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B1_V3 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [23]:
agent_b1_v3 = build_agent(model, "v3", output_type=ClassifierOutputB1, prompt_registry=PROMPT_REGISTRY_B1)

df_b1_exp3 = await run_experiment(
    agent_b1_v3, df_anotado,
    use_n1_context=False, concurrency=1,
    output_path="../results/b1_exp3_promptv3_qwen9b.csv",
    desc="B1 Exp3 - Prompt v3",
)
df_b1_exp3.head(3)

  Reanudando: 170/170 registros ya clasificados


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s
0,0,boe,Anuncio de la Confederación Hidrográfica del G...,AGU_GEN,True,AGU_GEN,cuenca_guadiana,NaN,True,anuncio,"[""AGU_GEN""]","[""cuenca_guadiana""]",0.95,"El texto menciona ""Concesión de aguas subterrá...",149.189
1,1,boe,Anuncio de formalización de contratos de: Pres...,AGU_GEN,False,NaN,NaN,NaN,False,anuncio,[],[],1.00,La publicación es un anuncio de formalización ...,65.713
2,2,boe,Anuncio de formalización de contratos de: Pres...,AGU_GEN,False,NaN,NaN,NaN,False,anuncio,[],[],0.95,"La publicación es un ""Anuncio de formalización...",74.844


In [24]:
df_b1_exp3 = pd.read_csv("../results/b1_exp3_promptv3_qwen9b.csv")
df_eval_b1_3 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b1_exp3[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b1_3 = compute_metrics_B1(df_eval_b1_3, "Experimento B1-3 - Prompt v3")

-- Experimento B1-3 - Prompt v3 --

is_relevant  Acc=0.818  P=0.818  R=1.000  F1=0.900
             TP=139  FP=31  FN=0  TN=0

N2 multilabel:
  Micro F1:      0.852
  Macro F1:      0.842
  Hamming Loss:  0.0257
  Jaccard:       0.695
  Subset Acc:    0.794

Label           P      R     F1   Sup
-----------------------------------
AGU_GEN     0.500  0.722  0.591    18
AGU_RIE     1.000  0.903  0.949    31
AGU_SND     0.842  0.842  0.842    19
AGU_ABS     0.955  1.000  0.977    21
AGU_IND     0.667  0.857  0.750     7
VIA_PEC     1.000  0.952  0.976    21
MON         0.900  0.900  0.900    10
ESP_NAT     0.417  1.000  0.588     5
RES         0.846  1.000  0.917    11
VER         0.636  1.000  0.778     7
PHD         1.000  1.000  1.000     2
-----------------------------------
Macro                     0.842

Subset Acc por cardinalidad:
  card=0 (no relevante) : 0.871  (27/31)
  card=1                : 0.789  (101/128)
  card=2                : 0.778  (7/9)
  card>=3               : 0.

In [25]:
print_errors(df_eval_b1_3, m_b1_3["exact"], m_b1_3["rel"], label="B1 Experimento 3 - Prompt v3")

-- Errores N2 en relevantes · B1 Experimento 3 - Prompt v3 --
Total: 31

ID 4 | GT=['AGU_RIE'] | PRED=['AGU_GEN', 'AGU_RIE']
  Falta: [] | Sobra: ['AGU_GEN']
  Anuncio de la Confederación Hidrográfica del Duero, O.A., de información pública del expediente de m...
  Razonamiento: El texto menciona "concesión de un aprovechamiento de aguas subterráneas" sin especificar uso explícito, lo que activa AGU_GEN por defecto residual. A

ID 7 | GT=['AGU_GEN'] | PRED=[]
  Falta: ['AGU_GEN'] | Sobra: []
  Anuncio de acuerdo de información pública de la Comisaría de Aguas de la Confederación Hidrográfica ...
  Razonamiento: La publicación es un anuncio sobre "Aprovechamiento de pastos" en el término municipal de Puebla de Lillo (León), pero no se detectan categorías N2: n

ID 8 | GT=['AGU_RIE'] | PRED=['AGU_GEN']
  Falta: ['AGU_RIE'] | Sobra: ['AGU_GEN']
  Anuncio de la Confederación Hidrográfica del Duero, O.A., de información pública del expediente de m...
  Razonamiento: El texto menciona "aprov

---

##  9. Experimento 4 - Prompt v4

**Problema**: el Exp 3 introduce 4 regresiones: (1) ESP_NAT se dispara por nombres de organismos que contienen "Paisaje", "Medio Natural" o "Desarrollo Sostenible", (2) VER se dispara por "alcantarillado" como infraestructura, (3) AGU_RIE no se detecta cuando la fuente es "aguas subterraneas" sin captacion explicita, (4) AGU_GEN no reconocido para aprovechamientos en DPH sin concesion de agua.

**Objetivo**: verificar si las 4 reglas correctoras del prompt V4 eliminan estas regresiones sin introducir nuevos errores.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B1_V4 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [26]:
agent_b1_v4 = build_agent(model, "v4", output_type=ClassifierOutputB1, prompt_registry=PROMPT_REGISTRY_B1)

df_b1_exp4 = await run_experiment(
    agent_b1_v4, df_anotado,
    use_n1_context=False, concurrency=1,
    output_path="../results/b1_exp4_promptv4_qwen9b.csv",
    desc="B1 Exp4 - Prompt v4",
)
df_b1_exp4.head(3)

  Reanudando: 44/170 registros ya clasificados


B1 Exp4 - Prompt v4:   1%|          | 1/126 [01:49<3:48:00, 109.45s/it]


  [ERROR] UnexpectedModelBehavior: Exceeded maximum output retries (1)


B1 Exp4 - Prompt v4: 100%|██████████| 126/126 [2:14:39<00:00, 64.12s/it] 

Tiempo: 8078.5s total  |  64.11s/item  |  126 items


,id,bulletin,description,grupo_muestreo,is_relevant_gt,categories_gt,subcategories_gt,notas_anotador,is_relevant_pred,act_type_pred,categories_pred,subcategories_pred,confidence,reasoning,duration_s
0,0,boe,Anuncio de la Confederación Hidrográfica del G...,AGU_GEN,True,AGU_GEN,cuenca_guadiana,NaN,True,anuncio,"[""AGU_GEN""]",[],0.55,"La descripción menciona ""concesión de aguas su...",249.912
1,1,boe,Anuncio de formalización de contratos de: Pres...,AGU_GEN,False,NaN,NaN,NaN,False,anuncio,[],[],1.00,Es un anuncio de formalización de contrato de ...,65.593
2,2,boe,Anuncio de formalización de contratos de: Pres...,AGU_GEN,False,NaN,NaN,NaN,False,anuncio,[],[],1.00,"La publicación se refiere a un ""Contrato de se...",69.597


In [27]:
df_b1_exp4 = pd.read_csv("../results/b1_exp4_promptv4_qwen9b.csv")
df_eval_b1_4 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b1_exp4[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b1_4 = compute_metrics_B1(df_eval_b1_4, "Experimento B1-4 - Prompt v4")

-- Experimento B1-4 - Prompt v4 --

is_relevant  Acc=0.818  P=0.818  R=1.000  F1=0.900
             TP=139  FP=31  FN=0  TN=0

N2 multilabel:
  Micro F1:      0.879
  Macro F1:      0.895
  Hamming Loss:  0.0198
  Jaccard:       0.699
  Subset Acc:    0.824

Label           P      R     F1   Sup
-----------------------------------
AGU_GEN     0.577  0.833  0.682    18
AGU_RIE     1.000  0.871  0.931    31
AGU_SND     0.850  0.895  0.872    19
AGU_ABS     0.944  0.810  0.872    21
AGU_IND     1.000  1.000  1.000     7
VIA_PEC     1.000  0.952  0.976    21
MON         0.800  0.800  0.800    10
ESP_NAT     0.714  1.000  0.833     5
RES         0.917  1.000  0.957    11
VER         1.000  0.857  0.923     7
PHD         1.000  1.000  1.000     2
-----------------------------------
Macro                     0.895

Subset Acc por cardinalidad:
  card=0 (no relevante) : 0.935  (29/31)
  card=1                : 0.812  (104/128)
  card=2                : 0.778  (7/9)
  card>=3               : 0.

In [28]:
print_errors(df_eval_b1_4, m_b1_4["exact"], m_b1_4["rel"], label="B1 Experimento 4 - Prompt v4")

-- Errores N2 en relevantes · B1 Experimento 4 - Prompt v4 --
Total: 28

ID 8 | GT=['AGU_RIE'] | PRED=['AGU_RIE', 'AGU_SND']
  Falta: [] | Sobra: ['AGU_SND']
  Anuncio de la Confederación Hidrográfica del Duero, O.A., de información pública del expediente de m...
  Razonamiento: Se menciona "expediente de modificación de características de concesión" y el texto incluye explícitamente "concesión de un aprovechamiento de aguas s

ID 9 | GT=['AGU_GEN'] | PRED=[]
  Falta: ['AGU_GEN'] | Sobra: []
  Anuncio de la Confederación Hidrográfica del Tajo O.A. de la resolución de la solicitud de una conce...
  Razonamiento: El texto describe una "concesión de aguas en el término municipal de Utande (Guadalajara)" pero no especifica ningún uso concreto del agua. Según las 

ID 12 | GT=['AGU_GEN'] | PRED=[]
  Falta: ['AGU_GEN'] | Sobra: []
  Anuncio de la Confederación Hidrográfica del Miño-Sil, O.A. por el que se publica el inicio del trám...
  Razonamiento: La publicación trata sobre un área recrea

---

##  10. Experimento 5 - Prompt v5 (few-shot)

**Problema**: algunos patrones persisten a pesar de las reglas explicitas: el modelo no generaliza correctamente AGU_RIE vs AGU_SND/GEN, ESP_NAT con organismos ambientales, y VER con infraestructura de saneamiento.

**Objetivo**: comprobar si 5 ejemplos few-shot quirurgicos sobre los errores mas persistentes mejoran el rendimiento respecto a V4.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B1_V5 (V4 + 5 ejemplos) · few-shot · sin contexto N1.

**Resultados**: pendiente.

In [29]:
agent_b1_v5 = build_agent(model, "v5", output_type=ClassifierOutputB1, prompt_registry=PROMPT_REGISTRY_B1)

df_b1_exp5 = await run_experiment(
    agent_b1_v5, df_anotado,
    use_n1_context=False, concurrency=1,
    output_path="../results/b1_exp5_promptv5_qwen9b.csv",
    desc="B1 Exp5 - Prompt v5 few-shot",
)
df_b1_exp5.head(3)

B1 Exp5 - Prompt v5 few-shot:   4%|▍         | 7/170 [07:35<2:55:22, 64.56s/it]


  [ERROR] UnexpectedModelBehavior: Model token limit (provider default) exceeded before any response was generated. Increase the `max_tokens` model setting, or simplify the prompt to result in a shorter response that will fit within the limit.


B1 Exp5 - Prompt v5 few-shot:   5%|▌         | 9/170 [10:04<3:07:16, 69.79s/it]


  [ERROR] UnexpectedModelBehavior: Model token limit (provider default) exceeded before any response was generated. Increase the `max_tokens` model setting, or simplify the prompt to result in a shorter response that will fit within the limit.


B1 Exp5 - Prompt v5 few-shot:   6%|▌         | 10/170 [11:23<3:13:48, 72.68s/it]

...

B1 Exp5 - Prompt v5 few-shot:  44%|████▎     | 74/170 [1:12:58<1:42:04, 63.79s/it]


  [ERROR] UnexpectedModelBehavior: Model token limit (provider default) exceeded before any response was generated. Increase the `max_tokens` model setting, or simplify the prompt to result in a shorter response that will fit within the limit.


B1 Exp5 - Prompt v5 few-shot:  45%|████▍     | 76/170 [1:16:23<1:34:28, 60.31s/it]


CancelledError: 


  [ERROR] ClosedResourceError: 


In [ ]:
df_b1_exp5 = pd.read_csv("../results/b1_exp5_promptv5_qwen9b.csv")
df_eval_b1_5 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b1_exp5[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b1_5 = compute_metrics_B1(df_eval_b1_5, "Experimento B1-5 - Prompt v5 few-shot")

In [ ]:
print_errors(df_eval_b1_5, m_b1_5["exact"], m_b1_5["rel"], label="B1 Experimento 5 - Prompt v5 few-shot")

---

##  11. Comparativa de modelos - Gemma 4B con prompt V4

**Problema**: todos los experimentos anteriores usan Qwen 3.5 9B. No sabemos si el prompt es transferible a modelos mas pequeños y rapidos.

**Objetivo**: comprobar si Gemma 4 4B con el mejor prompt (V4) alcanza un rendimiento comparable al 9B, midiendo transferibilidad y coste en tiempo.

**Enfoque**: Gemma 4 4B · SYSTEM_PROMPT_B1_V4 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider as OAIProvider

# Cargar gemma-4-e4b-it en LM Studio antes de ejecutar
model_gemma = OpenAIChatModel("gemma-4-e4b-it", provider=OAIProvider(base_url="http://localhost:1234/v1", api_key="lm-studio"))
agent_b1_gemma = build_agent(model_gemma, "v4", output_type=ClassifierOutputB1, prompt_registry=PROMPT_REGISTRY_B1)

df_b1_gemma = await run_experiment(
    agent_b1_gemma, df_anotado,
    use_n1_context=False, concurrency=1,
    output_path="../results/b1_exp_gemma4b_v4.csv",
    desc="B1 Gemma4B - Prompt v4",
)
df_b1_gemma.head(3)

In [ ]:
df_b1_gemma = pd.read_csv("../results/b1_exp_gemma4b_v4.csv")
df_eval_b1_gemma = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b1_gemma[["description","is_relevant_pred","act_type_pred","categories_pred",
                 "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b1_gemma = compute_metrics_B1(df_eval_b1_gemma, "B1 Gemma 4B - Prompt v4")

In [ ]:
print_errors(df_eval_b1_gemma, m_b1_gemma["exact"], m_b1_gemma["rel"], label="B1 Gemma 4B - Prompt v4")

---

##  10. Tabla resumen - Comparativa de experimentos B1

In [ ]:
experimentos_cfg_B1 = [
    ("B1 Exp1 - Baseline V1",    "Qwen 3.5 9B", "Zero-shot", "../results/b1_exp1_baseline_qwen9b.csv"),
    ("B1 Exp2 - Prompt V2",      "Qwen 3.5 9B", "Zero-shot", "../results/b1_exp2_promptv2_qwen9b.csv"),
    ("B1 Exp3 - Prompt V3",      "Qwen 3.5 9B", "Zero-shot", "../results/b1_exp3_promptv3_qwen9b.csv"),
    ("B1 Exp4 - Prompt V4",      "Qwen 3.5 9B", "Zero-shot", "../results/b1_exp4_promptv4_qwen9b.csv"),
    ("B1 Exp5 - Gemma 4B V4",    "Gemma 4 4B",  "Zero-shot", "../results/b1_exp_gemma4b_v4.csv"),
]

rows_b1 = []
metrics_list_b1 = []
for nombre, modelo, config, path in experimentos_cfg_B1:
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","categories_gt","description"]].merge(
        df_r[["description","is_relevant_pred","categories_pred","confidence","duration_s"]],
        on="description", how="left"
    )
    m = compute_metrics_B1(df_e, verbose=False)
    metrics_list_b1.append({"Experimento": nombre, **m})
    rows_b1.append({
        "Experimento": nombre,
        "Modelo":      modelo,
        "Config":      config,
        "is_rel_f1":       m["is_rel_f1"],
        "micro_f1":        m["micro_f1"],
        "macro_f1":        m["macro_f1"],
        "hamming_loss":    m["hamming_loss"],
        "jaccard_samples": m["jaccard_samples"],
        "subset_accuracy": m["subset_accuracy"],
        "mean_duration_s": round(df_r["duration_s"].mean(), 2) if "duration_s" in df_r.columns else None,
        "total_duration_s": round(df_r["duration_s"].sum(), 0) if "duration_s" in df_r.columns else None,
    })

df_summary_b1 = pd.DataFrame(rows_b1)

# Formatear para display
df_display_b1 = df_summary_b1.copy()
df_display_b1.columns = [
    "Experimento", "Modelo", "Config",
    "is_rel F1", "Micro F1", "Macro F1", "Hamming", "Jaccard", "Subset Acc",
    "s/item", "Total (s)",
]
display(df_display_b1.set_index("Experimento"))